# TA library evaluation — pandas_ta vs ta vs TA-Lib

Which of the three technical-analysis libraries should `src/tools/ta_tools/` wrap?

Judged on two criteria:

1. **Coverage** — how many indicators, in which categories.
2. **API shape** — Series in / Series out, index preservation, NaN warm-up, naming.

Then a third section on **fitness for the Pine ports**, because coverage counts say
nothing about whether a library supplies the specific primitives `pine_scripts/`
is built from.

Runs offline on a seeded OHLC fixture, so it reproduces anywhere.


In [1]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with `pip install -e .`.
# On Colab the install runs here for the dependencies, but an editable install
# registers itself through a .pth file that Python reads only at interpreter
# startup, so the repo also goes on sys.path for this kernel.
import os
import sys

REPO = '/content/drive/MyDrive/github/quant_dev'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        raise FileNotFoundError(f'No repo at {REPO}. Update REPO above.')
    !pip install -q -e "{REPO}[ta]"
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

print(f'Running in Colab: {IN_COLAB}')

Running in Colab: False


In [2]:
import importlib.metadata as meta
import inspect
import warnings

import numpy as np
import pandas as pd

# Distinct aliases on purpose. The older test_ta_packages.ipynb rebound the name
# `ta` in three separate cells, so its three comparisons could never coexist in a
# single kernel run - which is much of why it never reached a conclusion.
import talib
import ta as ta_pkg
import pandas_ta as pta

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)


def version(module, dist):
    """Version from the module if it exposes one, else from package metadata."""
    v = getattr(module, '__version__', None)
    if v:
        return v
    try:
        return meta.version(dist) + '  (no __version__ attribute)'
    except Exception:
        return 'unknown'


ENV = pd.DataFrame([
    {'library': 'TA-Lib',    'import as': 'talib',  'version': version(talib, 'TA-Lib')},
    {'library': 'ta',        'import as': 'ta_pkg', 'version': version(ta_pkg, 'ta')},
    {'library': 'pandas_ta', 'import as': 'pta',    'version': version(pta, 'pandas-ta')},
])
print(f'pandas {pd.__version__}   |   numpy {np.__version__}')
ENV

pandas 3.0.2   |   numpy 2.2.6


,library,import as,version
0,TA-Lib,talib,0.6.8
1,ta,ta_pkg,0.11.0 (no __version__ attribute)
2,pandas_ta,pta,0.4.71b0 (no __version__ attribute)


## Offline fixture

One seeded OHLC frame feeds every comparison, so no library is advantaged by
different data and the notebook needs no network.


In [3]:
def make_bars(n=500, seed=0, start='2023-01-02'):
    """Seeded OHLC bars: a GBM close, with open/high/low derived from it."""
    rng = np.random.default_rng(seed)
    close = 100 * np.cumprod(1 + rng.normal(0.0004, 0.011, n))
    reach = np.abs(rng.normal(0, 0.004, n))
    return pd.DataFrame(
        {
            'open':   np.r_[close[0], close[:-1]],
            'high':   close * (1 + reach),
            'low':    close * (1 - reach),
            'close':  close,
            'volume': rng.integers(1_000, 10_000, n).astype(float),
        },
        index=pd.bdate_range(start, periods=n),
    )


BARS = make_bars()
OPEN, HIGH, LOW, CLOSE = (BARS[c] for c in ('open', 'high', 'low', 'close'))
print(f'{len(BARS)} bars, {BARS.index[0]:%Y-%m-%d} to {BARS.index[-1]:%Y-%m-%d}')
BARS.head()

500 bars, 2023-01-02 to 2024-11-29


,open,high,low,close,volume
2023-01-02,100.178303,100.696383,99.660224,100.178303,8367.0
2023-01-03,100.178303,100.254401,99.891199,100.072800,8576.0
2023-01-04,100.072800,101.499400,100.136214,100.817807,8569.0
2023-01-05,100.817807,101.268584,100.680351,100.974468,4762.0
2023-01-06,100.974468,100.914871,99.924888,100.419879,4851.0


## 1. Coverage

Enumerated from each library rather than counted by hand.


In [4]:
def talib_coverage():
    return {g: sorted(fns) for g, fns in talib.get_function_groups().items()}


def ta_coverage():
    out = {}
    for group in ('momentum', 'trend', 'volatility', 'volume', 'others'):
        module = getattr(ta_pkg, group)
        out[group] = sorted(name for name, _ in
                            inspect.getmembers(module, inspect.isfunction)
                            if not name.startswith('_'))
    return out


def pta_coverage():
    return {k: sorted(v) for k, v in pta.Category.items()}


COVERAGE = {'TA-Lib': talib_coverage(), 'ta': ta_coverage(), 'pandas_ta': pta_coverage()}

counts = pd.DataFrame([
    {'library': lib, 'category': cat, 'count': len(names)}
    for lib, groups in COVERAGE.items() for cat, names in groups.items()
])

summary = (counts.groupby('library')['count']
           .agg(functions='sum', categories='size')
           .sort_values('functions', ascending=False))

# TA-Lib's headline number is inflated by candlestick pattern recognisers, which
# are a different kind of thing from an indicator. Show the count without them.
patterns = len(COVERAGE['TA-Lib'].get('Pattern Recognition', []))
summary['candlestick patterns'] = [patterns if lib == 'TA-Lib' else 0
                                   for lib in summary.index]
summary['functions ex-patterns'] = summary['functions'] - summary['candlestick patterns']
summary

,functions,categories,candlestick patterns,functions ex-patterns
library,,,,
TA-Lib,158,10,61,97
pandas_ta,151,9,0,151
ta,80,5,0,80


In [5]:
# Where those functions actually sit.
(counts.pivot_table(index='category', columns='library', values='count', fill_value=0)
       .astype(int)
       .sort_values('TA-Lib', ascending=False))

library,TA-Lib,pandas_ta,ta
category,,,
Pattern Recognition,61,0,0
Momentum Indicators,30,0,0
Overlap Studies,17,0,0
Math Transform,15,0,0
Math Operators,11,0,0
Statistic Functions,9,0,0
Cycle Indicators,5,0,0
Price Transform,4,0,0
Volatility Indicators,3,0,0


## 2. API shape

Same fixture, same indicators, three libraries. Recording what each call returns:
the type, whether the DatetimeIndex survives, and how many leading NaNs the
warm-up leaves behind.


In [6]:
def probe(call):
    """Run one indicator call and describe the result rather than the values."""
    try:
        out = call()
    except Exception as exc:
        return {'status': f'FAILED {type(exc).__name__}', 'returns': '-',
                'index kept': '-', 'warm-up NaNs': '-'}
    values = pd.Series(np.asarray(out, dtype='float64'))
    return {
        'status': 'ok',
        'returns': type(out).__name__,
        'index kept': bool(isinstance(out, pd.Series)
                           and out.index.equals(CLOSE.index)),
        'warm-up NaNs': int(values.isna().sum()),
    }


CASES = {
    ('SMA(20)', 'TA-Lib'):    lambda: talib.SMA(CLOSE, timeperiod=20),
    ('SMA(20)', 'ta'):        lambda: ta_pkg.trend.sma_indicator(CLOSE, window=20),
    ('SMA(20)', 'pandas_ta'): lambda: pta.sma(CLOSE, length=20),

    ('EMA(20)', 'TA-Lib'):    lambda: talib.EMA(CLOSE, timeperiod=20),
    ('EMA(20)', 'ta'):        lambda: ta_pkg.trend.ema_indicator(CLOSE, window=20),
    ('EMA(20)', 'pandas_ta'): lambda: pta.ema(CLOSE, length=20),

    ('ATR(14)', 'TA-Lib'):    lambda: talib.ATR(HIGH, LOW, CLOSE, timeperiod=14),
    ('ATR(14)', 'ta'):        lambda: ta_pkg.volatility.average_true_range(
                                          HIGH, LOW, CLOSE, window=14),
    ('ATR(14)', 'pandas_ta'): lambda: pta.atr(HIGH, LOW, CLOSE, length=14),

    ('RSI(14)', 'TA-Lib'):    lambda: talib.RSI(CLOSE, timeperiod=14),
    ('RSI(14)', 'ta'):        lambda: ta_pkg.momentum.rsi(CLOSE, window=14),
    ('RSI(14)', 'pandas_ta'): lambda: pta.rsi(CLOSE, length=14),

    ('LINREG(11)', 'TA-Lib'):    lambda: talib.LINEARREG(CLOSE, timeperiod=11),
    ('LINREG(11)', 'ta'):        lambda: ta_pkg.trend.linreg(CLOSE, 11),
    ('LINREG(11)', 'pandas_ta'): lambda: pta.linreg(CLOSE, length=11),
}

api = pd.DataFrame(
    [{'indicator': ind, 'library': lib, **probe(fn)}
     for (ind, lib), fn in CASES.items()]
).set_index(['indicator', 'library'])
api

status returns index kept warm-up NaNs
indicator  library                                                         
SMA(20)    TA-Lib                        ok  Series       True           19
           ta                            ok  Series       True           19
           pandas_ta                     ok  Series       True           19
EMA(20)    TA-Lib                        ok  Series       True           19
           ta                            ok  Series       True           19
           pandas_ta                     ok  Series       True           19
ATR(14)    TA-Lib                        ok  Series       True           14
           ta                            ok  Series       True            0
           pandas_ta                     ok  Series       True           14
RSI(14)    TA-Lib                        ok  Series       True           14
           ta                            ok  Series       True           13
           pandas_ta                     ok  Series       True           14
LINREG(11) TA-Lib                        ok  Series       True           10
           ta         FAILED AttributeError       -          -            -
           pandas_ta                     ok  Series       True           10

In [7]:
# pandas_ta's distinguishing feature is a DataFrame accessor; the others are
# module-level functions only.
try:
    accessor = BARS.ta.sma(length=20)
    print(f'BARS.ta.sma(length=20) -> {type(accessor).__name__}, '
          f'name={accessor.name!r}, len={len(accessor)}')
except Exception as exc:
    print(f'DataFrame accessor FAILED: {type(exc).__name__}: {exc}')

BARS.ta.sma(length=20) -> Series, name='SMA_20', len=500


## 3. Fitness for the Pine ports

Coverage counts do not answer the question that decides this: can the library
express the primitives the Pine scripts are built from? Probed by resolving each
name, not asserted from documentation.


In [8]:
def resolve(module, dotted):
    """Return the attribute at a dotted path, or None if any step is missing."""
    obj = module
    for part in dotted.split('.'):
        obj = getattr(obj, part, None)
        if obj is None:
            return None
    return obj


def mark(module, dotted):
    return dotted if resolve(module, dotted) is not None else '—'


DASH = '—'
NEEDS = [
    ('sma',               mark(talib, 'SMA'),
                          mark(ta_pkg, 'trend.sma_indicator'),
                          mark(pta, 'sma')),
    ('ema',               mark(talib, 'EMA'),
                          mark(ta_pkg, 'trend.ema_indicator'),
                          mark(pta, 'ema')),
    ('rma / Wilder',      DASH + '  (internal to ATR, RSI)',
                          mark(ta_pkg, 'trend.rma'),
                          mark(pta, 'rma')),
    ('atr',               mark(talib, 'ATR'),
                          mark(ta_pkg, 'volatility.average_true_range'),
                          mark(pta, 'atr')),
    ('stdev',             mark(talib, 'STDDEV'),
                          mark(ta_pkg, 'volatility.stdev'),
                          mark(pta, 'stdev')),
    ('linreg value',      mark(talib, 'LINEARREG'),
                          mark(ta_pkg, 'trend.linreg'),
                          mark(pta, 'linreg')),
    ('linreg slope',      mark(talib, 'LINEARREG_SLOPE'),
                          mark(ta_pkg, 'trend.linreg_slope'),
                          mark(pta, 'linreg_slope')),
    ('linreg intercept',  mark(talib, 'LINEARREG_INTERCEPT'),
                          mark(ta_pkg, 'trend.linreg_intercept'),
                          mark(pta, 'linreg_intercept')),
    ('pivot high/low + publication delay',
                          DASH, DASH, DASH + '  (`pivots` is S/R levels)'),
    ('harness for Pine `var` recursion',
                          DASH, DASH, DASH),
]

pine = pd.DataFrame(NEEDS, columns=['Pine primitive', 'TA-Lib', 'ta', 'pandas_ta'])
pine.set_index('Pine primitive')

,TA-Lib,ta,pandas_ta
Pine primitive,,,
sma,SMA,trend.sma_indicator,sma
ema,EMA,trend.ema_indicator,ema
rma / Wilder,"— (internal to ATR, RSI)",—,rma
atr,ATR,volatility.average_true_range,atr
stdev,STDDEV,—,stdev
linreg value,LINEARREG,—,linreg
linreg slope,LINEARREG_SLOPE,—,—
linreg intercept,LINEARREG_INTERCEPT,—,—
pivot high/low + publication delay,—,—,— (`pivots` is S/R levels)


In [9]:
# Pine's ta.linreg(src, length, offset) evaluates the fitted line `offset` bars
# back:  intercept + slope * (length - 1 - offset).
# TA-Lib's LINEARREG is that line at the current bar, so Pine's offset
# reconstructs exactly as LINEARREG - LINEARREG_SLOPE * offset.
N = 11
value = talib.LINEARREG(CLOSE, timeperiod=N)
slope = talib.LINEARREG_SLOPE(CLOSE, timeperiod=N)

recon = pd.DataFrame({f'offset={k}': value - slope * k for k in (0, 1, 3)})
print('Pine linreg reconstructed from TA-Lib:')
display(recon.tail(3))

# The trap: pandas_ta also takes an `offset`, but it means something else -
# a post-shift of the output series, not where the fitted line is evaluated.
base = pta.linreg(CLOSE, length=N)
shifted = pta.linreg(CLOSE, length=N, offset=3)
same_as_shift = bool(shifted.iloc[20:].round(10).equals(base.shift(3).iloc[20:].round(10)))
print(f'\npandas_ta offset=3 is just the output shifted 3 bars: {same_as_shift}')
print('  -> not Pine semantics; do not use it for the ports.')

Pine linreg reconstructed from TA-Lib:


,offset=0,offset=1,offset=3
2024-11-27,103.094493,103.190801,103.383416
2024-11-28,102.356688,102.579344,103.024657
2024-11-29,102.099685,102.331988,102.796594



pandas_ta offset=3 is just the output shifted 3 bars: True
  -> not Pine semantics; do not use it for the ports.


## 4. Scorecard


In [10]:
def covered(column):
    """How many Pine primitives this library supplies outright."""
    return sum(1 for row in NEEDS if not row[column].startswith(DASH))


scorecard = pd.DataFrame([
    {'library': 'TA-Lib',
     'indicators (ex-patterns)': int(summary.loc['TA-Lib', 'functions ex-patterns']),
     'Pine primitives': f'{covered(1)}/{len(NEEDS)}',
     'DataFrame accessor': 'no',
     'maintenance risk': 'stable; C extension already built here'},
    {'library': 'ta',
     'indicators (ex-patterns)': int(summary.loc['ta', 'functions ex-patterns']),
     'Pine primitives': f'{covered(2)}/{len(NEEDS)}',
     'DataFrame accessor': 'no',
     'maintenance risk': 'pure python, stable'},
    {'library': 'pandas_ta',
     'indicators (ex-patterns)': int(summary.loc['pandas_ta', 'functions ex-patterns']),
     'Pine primitives': f'{covered(3)}/{len(NEEDS)}',
     'DataFrame accessor': 'yes',
     'maintenance risk': 'beta (0.4.x), not validated against pandas 3'},
]).set_index('library')
scorecard

,indicators (ex-patterns),Pine primitives,DataFrame accessor,maintenance risk
library,,,,
TA-Lib,97,7/10,no,stable; C extension already built here
ta,80,3/10,no,"pure python, stable"
pandas_ta,151,6/10,yes,"beta (0.4.x), not validated against pandas 3"


## Decision

**Wrap TA-Lib.**

Reading the three sections together:

- **Coverage** is close once candlestick pattern recognisers are set aside. The
  headline counts flatter TA-Lib, since a large share of its functions are pattern
  recognisers rather than indicators.
- **API shape** is also close. All three accept a Series and return a Series with
  the index preserved. The one real divergence is ATR warm-up, where `ta` returns
  no leading NaNs while the other two do — it seeds differently, which would
  quietly change any warm-up-sensitive calculation. `pandas_ta` alone offers a
  DataFrame accessor, which is genuinely pleasant but not decisive.
- **Fitness for the Pine ports** breaks the tie. `ta` has no linear regression at
  all, and LinReg Candles calls it four times while ZLSMA nests it — that rules
  `ta` out on this project's own terms. Between the other two, TA-Lib exposes
  `LINEARREG`, `LINEARREG_SLOPE` and `LINEARREG_INTERCEPT`, which is exactly what
  makes Pine's `offset` semantics reconstructible. `pandas_ta` exposes `linreg` but
  no slope or intercept, and its `offset` argument means something different.

The deciding practical factor is risk: `pandas_ta` 0.4.71b0 is a beta running on
pandas 3.0.2, a combination its authors have not validated. TA-Lib's C extension is
already built and working in this environment — which is worth noting, since the
README had assumed the opposite when it put these libraries behind an optional
extra.

**One caveat to state plainly.** No library supplies pivot high/low with
publication delay, or a harness for Pine `var` series that depend on their own
previous bar. Those must be implemented regardless of which library is wrapped, so
`ta_tools` will be part wrapper and part original code. `backend.py` isolates the
wrapped half and `CAPABILITIES` records which primitives come from where.
